In [4]:
%pip install xgboost -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
from tqdm import tqdm
import numpy as np

In [7]:
df_txn = pd.read_csv("../data/acct_transaction.csv")
df_alert = pd.read_csv("../data/acct_alert.csv")

In [8]:
acct_features = (
    df_txn.groupby("from_acct")
    .agg(
        txn_count=("txn_amt", "count"),  # 交易筆數
        txn_sum=("txn_amt", "sum"),  # 總交易金額
        txn_mean=("txn_amt", "mean"),  # 平均交易金額
        txn_max=("txn_amt", "max"),  # 最大金額
        to_acct_unique=("to_acct", "nunique"),  # 收款帳戶數
        channel_unique=("channel_type", "nunique"),  # 使用通路數
    )
    .reset_index()
)

df_alert = df_alert.rename(columns={"acct": "from_acct"})

## 特徵工程

In [ ]:
print("合併資料...")
train_data = acct_features.merge(df_alert, on="from_acct", how="left")

print("製作可疑帳戶標籤...")
train_data["label"] = train_data["event_date"].notna().astype(int)
train_data["event_date"] = train_data["event_date"].fillna(-1).astype(int)


print("加入大金額判斷...")
acct_features["txn_amt_std"] = df_txn.groupby("from_acct")["txn_amt"].std().fillna(0)
acct_features["txn_large_ratio"] = (
    df_txn.assign(is_large=df_txn["txn_amt"] > 100000)
    .groupby("from_acct")["is_large"]
    .mean()
)


print("加入交易熵...")

# 計算每個帳戶對應的收款帳戶分布
counts = df_txn.groupby(["from_acct", "to_acct"]).size().reset_index(name="cnt")

# 計算每個帳戶的總交易數
total = counts.groupby("from_acct")["cnt"].transform("sum")

# 機率分布
counts["p"] = counts["cnt"] / total

# 熵公式：-Σ p log2 p
entropy_df = counts.groupby("from_acct").apply(
    lambda g: -(g["p"] * np.log2(g["p"])).sum()
)

# 塞回 features
acct_features["to_acct_entropy"] = entropy_df


print("加入通路類型...")
channel_dummies = pd.get_dummies(df_txn["channel_type"], prefix="channel")
df_txn = pd.concat([df_txn, channel_dummies], axis=1)

acct_channel = df_txn.groupby("from_acct")[channel_dummies.columns].mean()
acct_features = acct_features.merge(acct_channel, on="from_acct", how="left")


print("加入帳戶屬性特徵...")
acct_features["cross_bank_ratio"] = (
    df_txn.assign(cross=(df_txn["to_acct_type"] == "02"))
    .groupby("from_acct")["cross"]
    .mean()
)

print("加入時間特徵...")
acct_features["active_days"] = df_txn.groupby("from_acct")["txn_date"].nunique()
acct_features["night_ratio"] = (
    df_txn.assign(night=(df_txn["txn_time"].between(2200, 600)))
    .groupby("from_acct")["night"]
    .mean()
)

合併資料...
製作可疑帳戶標籤...
加入大金額判斷...
加入交易熵...


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# ========= 假設這裡已經有 train_data =========
# train_data 應該包含:
# - 特徵: txn_count, txn_sum, txn_mean, ...
# - 標籤: label
# - 其他欄位: from_acct, event_date (不作為特徵)

# 拿掉不能用的欄位 (acct id、日期)
X = train_data.drop(columns=["from_acct", "event_date", "label"], errors="ignore")
y = train_data["label"]

# ========= 切分訓練 / 驗證集 =========
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ========= 建立模型 =========
clf = RandomForestClassifier(
    n_estimators=100, max_depth=5, random_state=42, class_weight="balanced"
)
clf.fit(X_train, y_train)

# ========= 驗證 =========
y_proba = clf.predict_proba(X_val)[:, 1]
y_pred = (y_proba > 0.1).astype(int)

print("分類報告：")
print(classification_report(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_proba))

分類報告：
              precision    recall  f1-score   support

           0       0.00      0.00      0.00    163727
           1       0.00      1.00      0.00       153

    accuracy                           0.00    163880
   macro avg       0.00      0.50      0.00    163880
weighted avg       0.00      0.00      0.00    163880

ROC-AUC: 0.8566784473963533


c:\Users\bend0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\bend0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\bend0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [ ]:
from tqdm import tqdm
import xgboost as xgb
from xgboost.callback import TrainingCallback

class TQDMCallback(TrainingCallback):
    def __init__(self, total):
        self.pbar = tqdm(total=total, desc="Training XGBoost")

    def after_iteration(self, model, epoch, evals_log):
        self.pbar.update(1)
        return False  # False 表示不會中斷訓練

    def after_training(self, model):
        self.pbar.close()
        return model

# ========= 建立 DMatrix =========
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

params = {
    "objective": "binary:logistic",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": 1000,
    "eval_metric": "aucpr",
    "seed": 42,
}

num_round = 300
evals = [(dtrain, "train"), (dval, "val")]

# ========= 用自訂 Callback 加 tqdm =========
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=num_round,
    evals=evals,
    verbose_eval=False,
    callbacks=[TQDMCallback(num_round)]
)

# ========= 驗證 =========
y_proba = bst.predict(dval)
y_pred = (y_proba > 0.5).astype(int)

print("分類報告 (threshold=0.5)：")
print(classification_report(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_proba))
for th in [0.5, 0.3, 0.1]:
    y_pred_th = (y_proba > th).astype(int)
    print(f"\nThreshold = {th}")
    print(classification_report(y_val, y_pred_th))

Training XGBoost:   0%|          | 0/300 [00:00<?, ?it/s]

Training XGBoost: 100%|██████████| 300/300 [00:04<00:00, 63.24it/s]


分類報告 (threshold=0.5)：
              precision    recall  f1-score   support

           0       1.00      0.94      0.97    163727
           1       0.01      0.40      0.01       153

    accuracy                           0.94    163880
   macro avg       0.50      0.67      0.49    163880
weighted avg       1.00      0.94      0.97    163880

ROC-AUC: 0.7735787147032696

Threshold = 0.5
              precision    recall  f1-score   support

           0       1.00      0.94      0.97    163727
           1       0.01      0.40      0.01       153

    accuracy                           0.94    163880
   macro avg       0.50      0.67      0.49    163880
weighted avg       1.00      0.94      0.97    163880


Threshold = 0.3
              precision    recall  f1-score   support

           0       1.00      0.87      0.93    163727
           1       0.00      0.50      0.01       153

    accuracy                           0.87    163880
   macro avg       0.50      0.68      0.47 